# Milestone 4 — Vision Model Training (YOLO11m)

Trains and evaluates the Damage Agent (YOLO11m, the Milestone 3 selection) against the Milestone 4 target metrics:

| Metric | Target |
|---|---|
| mAP@50 | ≥ 0.70 overall |
| mAP@50-95 | ≥ 0.50 overall |
| Per-class F1 | ≥ 0.65 all classes |
| Inference speed | report only |

**Note on inference speed:** the metrics table specifies 640px input, but this project's actual pipeline uses **1280px** (Milestone 2, data-driven weighted-mean sizing; confirmed again in the Milestone 3 probe). This notebook measures speed at both sizes and reports 1280px as the number that matches production.

**Setup on Kaggle:** attach your Milestone 2 processed dataset as a notebook input, enable a GPU accelerator (P100 or T4x2), and edit `SOURCE_ROOT` in the config cell below to match your dataset's mounted path.

## 1. Setup

In [1]:
!pip install -q ultralytics

import torch, time, json
from pathlib import Path
import pandas as pd

assert torch.cuda.is_available(), "Enable GPU in Settings → Accelerator before running."
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM: %.1f GB" % (torch.cuda.get_device_properties(0).total_memory / 1e9))

GPU: Tesla T4
VRAM: 15.6 GB


In [2]:
!ls /kaggle/input/

# EDIT to match your Milestone 2 dataset's mounted path
SOURCE_ROOT = Path("/kaggle/input/datasets/m4rcuseryx/vehide-dataset-preprocessed/vehide_processed")
print("Exists:", SOURCE_ROOT.exists())
!ls {SOURCE_ROOT}

datasets
Exists: True
checksums.txt	  damage.yaml  labels		    splits
class_stats.json  images       processing_log.json


In [3]:
# damage.yaml points at the local machine that produced it — rewrite path for the Kaggle mount
import yaml

with open(SOURCE_ROOT / "damage.yaml") as f:
    cfg = yaml.safe_load(f)
cfg["path"] = str(SOURCE_ROOT)

DATA_YAML = "/kaggle/working/damage.yaml"
with open(DATA_YAML, "w") as f:
    yaml.safe_dump(cfg, f)

CLASS_NAMES = cfg["names"]
print(open(DATA_YAML).read())

names:
- dent
- scratch
- crack
- broken_lamp
- shattered_glass
- flat_tyre
nc: 6
path: /kaggle/input/datasets/m4rcuseryx/vehide-dataset-preprocessed/vehide_processed
test: images/test
train: images/train
val: images/val



## 2. Baseline configuration

Starts from the hyperparameters validated in the Milestone 3 probe (`imgsz=1280`, `batch=4`, `AdamW`, `lr0=0.001`), with two changes made after reviewing the first full-scale attempt's epoch 1–16 curves:

- **`cls=0.5`** (was `2.0`). At `cls=2.0`, both the Milestone 3 probe and epochs 1–16 of this run showed the same signature: `cls_loss` barely declining (18.22 → 15.83 over 16 epochs) while precision fell without a compensating recall gain. `2.0` was carried over from the probe without being tuned; `0.5` is Ultralytics' own default and is the first thing to revert before trying anything more exotic.
- **`cos_lr=True`, `lrf=0.001`** (was linear decay, `lrf=0.01`). Cosine decay gives a smoother, typically more effective learning-rate schedule for the back half of a 50-epoch run than linear decay.

`imgsz=1280` and `batch=4` are unchanged — both are justified by measured VRAM usage (8.4–8.6 GB of 14.9 GB available in the flagged run) and the Milestone 2 weighted-mean sizing calculation; there's no reason to touch either.

In [4]:
from ultralytics import YOLO

BASELINE_ARGS = dict(
    data=DATA_YAML,
    epochs=50,
    imgsz=1280,
    batch=4,
    optimizer="AdamW",
    lr0=0.001,
    lrf=0.001,            # tighter final LR, paired with cos_lr
    cos_lr=True,           # was linear decay
    weight_decay=0.0005,
    warmup_epochs=3,
    cls=0.5,               # was 2.0 — reverting to Ultralytics default, see note above
    patience=15,
    seed=42,
    deterministic=True,
    plots=True,
    project="/kaggle/working/runs/m4",
)
print(json.dumps(BASELINE_ARGS, indent=2))

{
  "data": "/kaggle/working/damage.yaml",
  "epochs": 50,
  "imgsz": 1280,
  "batch": 4,
  "optimizer": "AdamW",
  "lr0": 0.001,
  "lrf": 0.001,
  "cos_lr": true,
  "weight_decay": 0.0005,
  "warmup_epochs": 3,
  "cls": 0.5,
  "patience": 15,
  "seed": 42,
  "deterministic": true,
  "plots": true,
  "project": "/kaggle/working/runs/m4"
}


## 3. Train YOLO11m

Runs as `yolo11m_baseline_v2` (a new folder, not overwriting the first attempt) so the two runs' curves can be compared directly if useful.

In [ ]:
t0 = time.time()
model_11m = YOLO("yolo11m.pt")
results_11m = model_11m.train(name="yolo11m_baseline_v2", **BASELINE_ARGS)
wall_11m = time.time() - t0
print(f"\nYOLO11m baseline v2: {wall_11m/60:.1f} min total, {wall_11m/50:.1f} s/epoch")

Ultralytics 8.4.105 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/damage.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1280, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.001, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo11m_baseline_v2-2, nbs=64, nm

## 3.1 Checkpoint diagnostic — run this while training is in progress or right after

Two things worth checking without waiting for the full 50 epochs:

1. **Label sanity**: confirm ground-truth boxes actually sit on real damage regions, ruling out a labeling/geometry bug as the cause of slow convergence, separate from the hyperparameter question.
2. **Curve checkpoint**: at epoch 15 and epoch 25, compare `cls_loss` against the flagged run's values (15.94 at epoch 15). A meaningfully lower value (for example, under 10) is the signal the `cls=0.5` change is working; if it's still tracking close to the old run's pace, that's the point to stop and investigate further rather than let it run to completion.

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

run_dir = results_11m.save_dir

fig, axes = plt.subplots(1, 3, figsize=(20, 7))
for ax, fname, title in zip(
    axes,
    ["train_batch0.jpg", "val_batch0_labels.jpg", "val_batch0_pred.jpg"],
    ["Train batch (ground truth)", "Val batch (ground truth)", "Val batch (current predictions)"],
):
    path = run_dir / fname
    if path.exists():
        ax.imshow(Image.open(path))
        ax.set_title(title)
        ax.axis("off")
    else:
        ax.set_title(f"{fname} not found yet")
        ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Quick mid-run check — read the results.csv written so far without waiting for training to finish.
# Re-run this cell any time; it just re-reads whatever epochs have completed.
import pandas as pd

csv_path = results_11m.save_dir / "results.csv"
if csv_path.exists():
    df = pd.read_csv(csv_path)
    df.columns = [c.strip() for c in df.columns]
    print(df[["epoch", "train/box_loss", "train/cls_loss", "train/dfl_loss",
             "metrics/precision(B)", "metrics/recall(B)",
             "metrics/mAP50(B)", "metrics/mAP50-95(B)"]].tail(10).to_string(index=False))
else:
    print("results.csv not written yet — wait for epoch 1 to finish.")

## 4. Evaluate against the Milestone 4 target metrics (test split)

In [ ]:
import numpy as np

F1_TARGET, MAP50_TARGET, MAP5095_TARGET = 0.65, 0.70, 0.50

def evaluate(weights_path, imgsz=1280):
    model = YOLO(str(weights_path))
    m = model.val(data=DATA_YAML, split="test", imgsz=imgsz, plots=True)
    rows = []
    for idx, ci in enumerate(m.box.ap_class_index):
        p, r = float(m.box.p[idx]), float(m.box.r[idx])
        f1 = 2 * p * r / (p + r) if (p + r) else 0.0
        rows.append({
            "class": CLASS_NAMES[int(ci)], "precision": p, "recall": r,
            "f1": f1, "mAP50": float(m.box.ap50[idx]), "mAP50_95": float(m.box.ap[idx]),
            "meets_F1_target": f1 >= F1_TARGET,
        })
    per_class = pd.DataFrame(rows).sort_values("f1", ascending=False)
    overall = {
        "mAP50": float(m.box.map50), "mAP50_95": float(m.box.map),
        "meets_mAP50_target": float(m.box.map50) >= MAP50_TARGET,
        "meets_mAP50_95_target": float(m.box.map) >= MAP5095_TARGET,
    }
    return model, per_class, overall

model_11m_best, pc_11m, overall_11m = evaluate(results_11m.save_dir / "weights" / "best.pt")

print("YOLO11m overall:", overall_11m)
print("\nYOLO11m per-class:")
pc_11m.round(3)

## 5. Inference speed (report only — measured at both 640px and 1280px)

640px matches the metrics table as written; 1280px matches what this project actually deploys. Report both, label which is authoritative.

In [ ]:
def measure_speed(model, imgsz, n_warmup=5, n_measure=30):
    test_imgs = sorted((SOURCE_ROOT / "images" / "test").glob("*.jpg"))[: n_warmup + n_measure]
    for img in test_imgs[:n_warmup]:
        model.predict(str(img), imgsz=imgsz, verbose=False)
    torch.cuda.synchronize(); t0 = time.time()
    for img in test_imgs[n_warmup:]:
        model.predict(str(img), imgsz=imgsz, verbose=False)
    torch.cuda.synchronize()
    elapsed = time.time() - t0
    return (elapsed / n_measure) * 1000, n_measure / elapsed  # ms/image, FPS

speed_rows = []
for sz, label in [(640, "640px (table spec)"), (1280, "1280px (production)")]:
    ms, fps = measure_speed(model_11m_best, sz)
    speed_rows.append({"model": "YOLO11m", "input": label, "ms_per_image": round(ms, 1), "fps": round(fps, 1)})

pd.DataFrame(speed_rows)

## 6. Target compliance summary

In [ ]:
summary = pd.DataFrame([
    {"model": "YOLO11m", **overall_11m, "min_per_class_F1": pc_11m["f1"].min(),
     "all_classes_meet_F1": bool(pc_11m["meets_F1_target"].all())},
])
print(summary.round(4).to_markdown(index=False))

weak_11m = pc_11m[~pc_11m["meets_F1_target"]]["class"].tolist()
print(f"\nClasses below F1 target: {weak_11m or 'none'}")

# CarDD contingency trigger (Milestone 2, Section 7.2): fires if any of the three
# minority/irregular-boundary classes miss the F1 target.
trigger_classes = {"shattered_glass", "flat_tyre", "crack"}
hit = trigger_classes & set(weak_11m)
print(f"CarDD contingency trigger: {'TRIGGERED (' + ', '.join(hit) + ')' if hit else 'not triggered'}")

## 7. Save everything for the Milestone 4 report

In [ ]:
OUT = Path("/kaggle/working/m4_outputs")
OUT.mkdir(exist_ok=True)

pc_11m.to_csv(OUT / "yolo11m_per_class.csv", index=False)
summary.to_csv(OUT / "overall_summary.csv", index=False)
pd.DataFrame(speed_rows).to_csv(OUT / "inference_speed.csv", index=False)

with open(OUT / "m4_results.json", "w") as f:
    json.dump({
        "yolo11m": {"overall": overall_11m, "per_class": pc_11m.to_dict(orient="records"),
                    "s_per_epoch": round(wall_11m / 50, 1)},
        "inference_speed": speed_rows,
        "targets": {"mAP50": MAP50_TARGET, "mAP50_95": MAP5095_TARGET, "per_class_F1": F1_TARGET},
    }, f, indent=2)

print("Saved to", OUT)
print("\nCopy the weights into the repo:")
print(f"  cp {results_11m.save_dir}/weights/best.pt models/best.pt")